# NDLOCR CLI (ver.2.1) on Google Colab

このノートブックは、国立国会図書館の [ndlocr_cli](https://github.com/ndl-lab/ndlocr_cli) をGoogle Colab上で動かすための環境構築および推論実行テンプレートです。

### 事前準備: GPUの有効化
メニューの「ランタイム」 > 「ランタイムのタイプを変更」から、**ハードウェア アクセラレータ**を `T4 GPU` または `A100 GPU` などに変更してください。

In [ ]:
# GPUが正しく認識されているか確認します
!nvidia-smi

## 1. リポジトリの取得とモデルのダウンロード

In [ ]:
import os
%cd /content

# リポジトリのクローン (submoduleも含めて取得)
!git clone --recursive https://github.com/ndl-lab/ndlocr_cli.git

%cd /content/ndlocr_cli

# dockerbuild.sh と同じようにモデルファイルをダウンロード
!wget -nc https://lab.ndl.go.jp/dataset/ndlocr_v2/text_recognition_lightning/resnet-orient2.ckpt -P ./submodules/text_recognition_lightning/models
!wget -nc https://lab.ndl.go.jp/dataset/ndlocr_v2/text_recognition_lightning/rf_author/model.pkl -P ./submodules/text_recognition_lightning/models/rf_author/
!wget -nc https://lab.ndl.go.jp/dataset/ndlocr_v2/text_recognition_lightning/rf_title/model.pkl -P ./submodules/text_recognition_lightning/models/rf_title/
!wget -nc https://lab.ndl.go.jp/dataset/ndlocr_v2/ndl_layout/ndl_retrainmodel.pth -P ./submodules/ndl_layout/models
!wget -nc https://lab.ndl.go.jp/dataset/ndlocr_v2/separate_pages_mmdet/epoch_180.pth -P ./submodules/separate_pages_mmdet/models

## 2. システムパッケージ (KyTea 等) のインストール
ColabのUbuntu環境に不足しているライブラリをインストールします。

In [ ]:
%cd /content
!apt-get update
!apt-get install -y libgl1-mesa-dev libglib2.0-0 zip git wget

# KyTeaのビルドとインストール
!wget http://www.phontron.com/kytea/download/kytea-0.4.7.tar.gz
!tar xzvf kytea-0.4.7.tar.gz
%cd kytea-0.4.7
!./configure
!make
!make install
!ldconfig

# 元のディレクトリに戻る
%cd /content/ndlocr_cli

## 3. Pythonライブラリのインストール
PyTorch、mmdet、mmcv およびプロジェクトの依存関係をインストールします。

In [ ]:
%cd /content/ndlocr_cli

# Dockerfileに準拠した PyTorch 2.1.1 のインストール
!pip install torch==2.1.1 torchvision==0.16.1 torchaudio==2.1.1 torchtext==0.16.1 --index-url https://download.pytorch.org/whl/cu121

# Requirementsのインストール
!pip install -r requirements.txt

# mmdet と mmcv のインストール
!pip install mmdet==3.3.0
!pip install mmcv==2.1.0 -f https://download.openmmlab.com/mmcv/dist/cu121/torch2.1/index.html

# ruby_prediction用の追加要件
!pip install -r ./submodules/ruby_prediction/requirements.txt

## 4. 推論の実行準備
入力画像を配置するフォルダを作成します。
このセルを実行した後、左側のファイルブラウザから `/content/ndlocr_cli/input_dir/img/` の中に処理したい画像（.jpg や .png など）をアップロードしてください。

In [ ]:
%cd /content/ndlocr_cli
!mkdir -p input_dir/img
!mkdir -p output_dir

print("==============================================")
print("左側のフォルダアイコンを開き、")
print("/content/ndlocr_cli/input_dir/img/")
print("ディレクトリの中に画像ファイルをアップロードしてください。")
print("==============================================")

## 5. 推論処理の実行
画像が配置できたら、以下のセルを実行してOCRを実行します。
`-i`オプションで画像も出力、`-x`オプションでXML形式でも出力します。

In [ ]:
%cd /content/ndlocr_cli
!python main.py infer input_dir output_dir -s s -i -x

print("\n--- 処理完了 ---")
print("結果は /content/ndlocr_cli/output_dir に保存されています。")

## 6. 結果の確認
出力されたテキストの内容を確認します。

In [ ]:
!ls -la output_dir
print("\n=== テキスト出力の内容 ===")
!cat output_dir/txt/*